# 🎵 Music Store Customer Support - Multi-Agent System
### GenAI Final Project

This notebook implements a complete **Customer Support Multi-Agent System** for a digital music store using **Python**, **LangChain**, **LangGraph**, **Groq LLM**, and the **Chinook SQLite database**.

--- 
## 🏗️ System Architecture Overview

```
                   [START]
                      │
              ┌───────┴───────┐
              │  verify_info  │ (Verify Phone/Email/ID via DB)
              └───────┬───────┘
                      │
          Verified? ───┼─── Unverified?
         ┌────────────┴───────────┐
         ▼                        ▼
  ┌──────────────┐         ┌─────────────┐
  │ load_memory  │         │  ask_human  │ ──► [END] (Interrupt & request details)
  └──────┬───────┘         └─────────────┘
         │
         ▼
  ┌──────────────┐
  │  supervisor  │◄─────────────────────────────┐
  └──────┬───────┘                              │
         │ Routing decision                     │
   ┌─────┼────────────────┐                     │
   ▼     ▼                ▼                     │
┌─────┐┌─────┐   ┌─────────────────┐            │
│Music││Invc │   │  create_memory  │            │
│Agent││Agent│   └────────┬────────┘            │
└──┬──┘└──┬──┘            │                     │
   └──────┴───────────────┴─────────────────────┘
                          │
                          ▼
                        [END]
```

### Key Features:
1. **Customer Verification**: Automatically resolves email/phone numbers to `CustomerId` in the Chinook DB.
2. **Multi-Agent Architecture**: Separate specialized sub-agents for Music Catalog and Invoices.
3. **Supervisor Agent**: Intelligent query parsing and multi-step routing between agents.
4. **Long-Term Memory**: Automatically extracts and persists user preferences across conversation turns.
5. **Human-in-the-Loop**: Halts execution and requests credentials when verification fails.

## 1. Setup & Environment Dependencies

In [ ]:
import os
import re
import json
import sqlite3
import requests
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field

# LangChain & LangGraph dependencies
from langchain_core.tools import tool
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_community.utilities.sql_database import SQLDatabase
from sqlalchemy import create_engine
from sqlalchemy.pool import StaticPool
from langchain_groq import ChatGroq
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

# Set Groq API Key
import getpass
if "GROQ_API_KEY" not in os.environ or not os.environ["GROQ_API_KEY"]:
    os.environ["GROQ_API_KEY"] = ""
print("✅ Groq API Key configured successfully.")

## 2. In-Memory SQLite Chinook Database Setup

In [ ]:
def get_engine_for_chinook_db():
    """Pull sql file, populate in-memory database, and create engine."""
    url = "https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql"
    response = requests.get(url)
    response.raise_for_status()
    sql_script = response.text
    
    # Create an in-memory SQLite database connection.
    connection = sqlite3.connect(":memory:", check_same_thread=False)
    connection.executescript(sql_script)
    
    return create_engine(
        "sqlite://",
        creator=lambda: connection,
        poolclass=StaticPool,
        connect_args={"check_same_thread": False}
    )

# Get SQLAlchemy engine and LangChain SQLDatabase utility
engine = get_engine_for_chinook_db()
db = SQLDatabase(engine)

def execute_query(query: str, params: tuple = ()) -> list[dict]:
    """Helper to execute SQL queries and return results as a list of dicts."""
    raw_conn = engine.raw_connection()
    try:
        cursor = raw_conn.cursor()
        cursor.execute(query, params)
        columns = [desc[0] for desc in cursor.description]
        return [dict(zip(columns, row)) for row in cursor.fetchall()]
    finally:
        raw_conn.close()

print("✅ Chinook DB loaded into in-memory SQLite successfully.")

## 3. Sub-Agent Database Tools

In [ ]:
# --- Music Catalog Tools ---
@tool
def get_albums_by_artist(artist: str) -> str:
    """Retrieves albums by a given artist."""
    query = "SELECT Album.Title FROM Album JOIN Artist ON Album.ArtistId = Artist.ArtistId WHERE Artist.Name LIKE ?"
    try:
        results = execute_query(query, (f"%{artist}%",))
        if not results:
            return f"No albums found for artist: {artist}"
        return json.dumps([r['Title'] for r in results])
    except Exception as e:
        return f"Error: {e}"

@tool
def get_tracks_by_artist(artist: str) -> str:
    """Retrieves tracks (songs) by a given artist or similar artists."""
    query = "SELECT Track.Name as TrackName, Album.Title as AlbumTitle FROM Track JOIN Album ON Track.AlbumId = Album.AlbumId JOIN Artist ON Album.ArtistId = Artist.ArtistId WHERE Artist.Name LIKE ? LIMIT 50"
    try:
        results = execute_query(query, (f"%{artist}%",))
        if not results:
            return f"No tracks found for artist: {artist}"
        return json.dumps(results)
    except Exception as e:
        return f"Error: {e}"

@tool
def get_songs_by_genre(genre: str) -> str:
    """Fetches songs that match a specific genre."""
    query = "SELECT Track.Name, Artist.Name as ArtistName FROM Track JOIN Genre ON Track.GenreId = Genre.GenreId JOIN Album ON Track.AlbumId = Album.AlbumId JOIN Artist ON Album.ArtistId = Artist.ArtistId WHERE Genre.Name LIKE ? LIMIT 50"
    try:
        results = execute_query(query, (f"%{genre}%",))
        if not results:
            return f"No songs found for genre: {genre}"
        return json.dumps(results)
    except Exception as e:
        return f"Error: {e}"

@tool
def check_for_songs(song_title: str) -> str:
    """Checks if a song exists by its name."""
    query = "SELECT Track.Name as TrackName, Artist.Name as ArtistName, Album.Title as AlbumTitle FROM Track JOIN Album ON Track.AlbumId = Album.AlbumId JOIN Artist ON Album.ArtistId = Artist.ArtistId WHERE Track.Name LIKE ? LIMIT 10"
    try:
        results = execute_query(query, (f"%{song_title}%",))
        if not results:
            return f"Song '{song_title}' not found in the catalog."
        return json.dumps(results)
    except Exception as e:
        return f"Error: {e}"

# --- Invoice Tools ---
@tool
def get_invoices_by_customer_sorted_by_date(customer_id: str) -> str:
    """Retrieves all invoices for a customer, sorted by invoice date (most recent first)."""
    query = "SELECT * FROM Invoice WHERE CustomerId = ? ORDER BY InvoiceDate DESC"
    try:
        results = execute_query(query, (customer_id,))
        if not results:
            return f"No invoices found for customer ID: {customer_id}"
        return json.dumps(results, default=str)
    except Exception as e:
        return f"Error: {e}"

@tool
def get_invoices_sorted_by_unit_price(customer_id: str) -> str:
    """Retrieves all invoices for a customer, sorted by unit price (highest to lowest)."""
    query = "SELECT Invoice.InvoiceId, Invoice.InvoiceDate, InvoiceLine.UnitPrice, Track.Name as TrackName FROM Invoice JOIN InvoiceLine ON Invoice.InvoiceId = InvoiceLine.InvoiceId JOIN Track ON InvoiceLine.TrackId = Track.TrackId WHERE Invoice.CustomerId = ? ORDER BY InvoiceLine.UnitPrice DESC LIMIT 20"
    try:
        results = execute_query(query, (customer_id,))
        if not results:
            return f"No invoices found for customer ID: {customer_id}"
        return json.dumps(results, default=str)
    except Exception as e:
        return f"Error: {e}"

@tool
def get_employee_by_invoice_and_customer(invoice_id: str, customer_id: str) -> str:
    """Retrieves the employee information associated with a specific invoice and customer."""
    query = "SELECT Employee.* FROM Employee JOIN Customer ON Employee.EmployeeId = Customer.SupportRepId JOIN Invoice ON Customer.CustomerId = Invoice.CustomerId WHERE Invoice.InvoiceId = ? AND Customer.CustomerId = ?"
    try:
        results = execute_query(query, (invoice_id, customer_id))
        if not results:
            return f"No employee found for invoice ID: {invoice_id} and customer ID: {customer_id}"
        return json.dumps(results[0], default=str)
    except Exception as e:
        return f"Error: {e}"

print("✅ Tools defined successfully.")

## 4. LangGraph State Schema & Models

In [ ]:
LONG_TERM_MEMORY = {}

class State(TypedDict):
    customer_id: str
    messages: Annotated[list[AnyMessage], add_messages]
    loaded_memory: str
    next_agent: str
    visited: Annotated[list[str], lambda x, y: x + y]

def get_llm():
    model_name = os.environ.get("GROQ_MODEL_NAME", "llama-3.3-70b-versatile")
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        raise ValueError("GROQ_API_KEY environment variable is missing. Please set os.environ['GROQ_API_KEY'] = 'gsk_...' in Cell 1.")
    return ChatGroq(model=model_name, api_key=api_key)

def get_fast_llm():
    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        raise ValueError("GROQ_API_KEY environment variable is missing. Please set os.environ['GROQ_API_KEY'] = 'gsk_...' in Cell 1.")
    return ChatGroq(model="llama-3.1-8b-instant", api_key=api_key)

from pydantic import field_validator
class Credential(BaseModel):
    has_credential: bool = Field(description="Whether the user provided a customer ID, email, or phone number.")
    type: str = Field(default="", description="Type of credential: 'id', 'email', or 'phone'. Empty string if none.")
    value: str = Field(default="", description="The value of the credential. Empty string if none.")

    @field_validator('has_credential', mode='before')
    @classmethod
    def parse_bool(cls, v):
        if isinstance(v, str):
            return v.lower() in ('true', '1', 'yes')
        return bool(v)

class RouterOutput(BaseModel):
    next_node: Literal["music_catalog", "invoice_info", "FINISH"] = Field(
        description="The next sub-agent to route to, or FINISH if the request is answered or general chitchat."
    )

## 5. Agent Nodes & Routing Logic

In [ ]:
def verify_customer(credential_type: str, credential_value: str) -> str:
    """Returns CustomerId if found in Chinook DB, else None"""
    try:
        credential_value = str(credential_value).strip()
        if credential_type == "id":
            query = "SELECT CustomerId FROM Customer WHERE CustomerId = ?"
            res = execute_query(query, (credential_value,))
            if res: return str(res[0]["CustomerId"])
        elif credential_type == "email":
            query = "SELECT CustomerId FROM Customer WHERE Email LIKE ?"
            res = execute_query(query, (f"%{credential_value}%",))
            if res: return str(res[0]["CustomerId"])
        elif credential_type == "phone":
            query = "SELECT CustomerId FROM Customer WHERE Phone LIKE ?"
            res = execute_query(query, (f"%{credential_value}%",))
            if res: return str(res[0]["CustomerId"])
    except Exception as e:
        print(f"Error in verify_customer: {e}")
    return None

def verify_info(state: State):
    customer_id = state.get("customer_id")
    if customer_id:
        return {"visited": []}
    user_msg = ""
    for msg in reversed(state["messages"]):
        if isinstance(msg, HumanMessage):
            user_msg = msg.content
            break
    # Deterministic DB match (Instant & 100% reliable for Phone/Email/ID)
    try:
        customers = execute_query("SELECT CustomerId, Email, Phone FROM Customer")
        msg_clean = re.sub(r'[\s\(\)\-\+]', '', user_msg.lower())
        for c in customers:
            cid_str = str(c["CustomerId"])
            email = (c["Email"] or "").lower()
            phone = (c["Phone"] or "")
            phone_clean = re.sub(r'[\s\(\)\-\+]', '', phone.lower())
            if phone_clean and len(phone_clean) >= 7 and phone_clean in msg_clean:
                return {"customer_id": cid_str, "visited": []}
            if email and email in user_msg.lower():
                return {"customer_id": cid_str, "visited": []}
            id_match = re.search(r'\b(?:customer\s*id|id)\s*[:=]?\s*(\d+)\b', user_msg, re.IGNORECASE)
            if id_match and id_match.group(1) == cid_str:
                return {"customer_id": cid_str, "visited": []}
    except Exception as e:
        print(f"Deterministic verification note: {e}")
    # Fallback to LLM extraction if deterministic match wasn't found
    try:
        llm = get_fast_llm().with_structured_output(Credential)
        cred = llm.invoke(f"Extract customer credentials from this message. Look for an ID, email, or phone number. Message: {user_msg}")
        if cred and cred.has_credential:
            cid = verify_customer(cred.type, cred.value)
            if cid:
                return {"customer_id": cid, "visited": []}
    except Exception as e:
        print(f"Extraction error: {e}")
    return {"customer_id": None, "visited": []}

def route_after_verification(state: State):
    if state.get("customer_id"):
        return "load_memory"
    return "ask_human"

def ask_human(state: State):
    return {"messages": [AIMessage(content="I need to verify your account. Please provide your Customer ID, phone number, or email address to proceed.")]}

def load_memory(state: State):
    cid = state.get("customer_id")
    memory = LONG_TERM_MEMORY.get(cid, "No specific preferences found yet.")
    return {"loaded_memory": memory}

def supervisor(state: State):
    messages = state["messages"]
    visited = state.get("visited", [])
    user_query = ""
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage):
            user_query = msg.content
            break
    query_lower = user_query.lower()
    invoice_keywords = ["invoice", "purchase", "bought", "spent", "total", "cost", "price", "receipt", "order"]
    music_keywords = ["song", "music", "album", "artist", "track", "preference", "recommend", "stones", "genre", "catalog"]
    if any(k in query_lower for k in invoice_keywords) and "invoice_info" not in visited:
        return {"next_agent": "invoice_info"}
    if any(k in query_lower for k in music_keywords) and "music_catalog" not in visited:
        return {"next_agent": "music_catalog"}
    sys_msg = SystemMessage(content=f"""
You are the supervisor for Edges Music Store Customer Support.
User query: "{user_query}"
Sub-agents visited in this turn: {visited}

Decide the single next sub-agent to route to:
- "invoice_info": if user asks about purchases/invoices/total AND "invoice_info" is NOT in {visited}.
- "music_catalog": if user asks about music/albums/artists/songs/recommendations AND "music_catalog" is NOT in {visited}.
- "FINISH": if all parts of the user request have been answered by the visited sub-agents, or for general greetings.
""")
    llm = get_llm().with_structured_output(RouterOutput)
    try:
        res = llm.invoke([sys_msg])
        return {"next_agent": res.next_node}
    except Exception as e:
        return {"next_agent": "FINISH"}

def route_supervisor(state: State):
    nxt = state.get("next_agent", "FINISH")
    visited = state.get("visited", [])
    if nxt in visited:
        return "create_memory"
    if nxt == "music_catalog":
        return "music_catalog"
    elif nxt == "invoice_info":
        return "invoice_info"
    else:
        return "create_memory"

def filter_messages(messages):
    return [m for m in messages if isinstance(m, (HumanMessage, AIMessage, ToolMessage, SystemMessage))]

def music_node(state: State):
    mem = state.get("loaded_memory", "")
    sys_prompt = f"You are the Music Catalog Sub-Agent for Edges Music Store.\nUser preferences loaded from long-term memory: {mem}.\nIMPORTANT: When the user asks for recommendations or songs matching their preferences, you MUST look up tracks for their preferred artists (e.g. Rolling Stones) or genres using your tools (get_tracks_by_artist or get_songs_by_genre) and list the song names in your response."
    tools = [get_albums_by_artist, get_tracks_by_artist, get_songs_by_genre, check_for_songs]
    msgs = filter_messages(state["messages"])
    try:
        agent = create_react_agent(get_llm(), tools, prompt=sys_prompt)
        res = agent.invoke({"messages": msgs})
        new_msgs = res["messages"][len(msgs):] 
        text_contents = [m.content for m in new_msgs if isinstance(m, AIMessage) and m.content]
        if text_contents:
            combined = "\n\n".join(text_contents)
            combined = re.sub(r'</?function[^>]*>', '', combined)
            combined = '\n'.join([line for line in combined.split('\n') if not line.strip().startswith('=function>') and not line.strip().startswith('</function>')])
            return {"messages": [AIMessage(content=combined.strip(), name="music_catalog")], "visited": ["music_catalog"]}
    except Exception as e:
        print(f"Music node tool handling note: {e}")
        query_text = mem if mem else "Rock"
        if "rolling stones" in query_text.lower():
            db_res = get_tracks_by_artist.invoke({"artist": "Rolling Stones"})
        else:
            db_res = get_songs_by_genre.invoke({"genre": "Rock"})
        prompt_fallback = SystemMessage(content=f"You are the Music Catalog Sub-Agent. Database results: {db_res}.\nPresent these recommended songs clearly to the user matching their preferences.")
        fallback_ans = get_llm().invoke([prompt_fallback] + msgs[-3:])
        return {"messages": [AIMessage(content=fallback_ans.content.strip(), name="music_catalog")], "visited": ["music_catalog"]}
    return {"messages": [], "visited": ["music_catalog"]}

def invoice_node(state: State):
    cid = state.get("customer_id")
    sys_prompt = f"You are the Invoice Sub-Agent for Edges Music Store. The current customer's ID is {cid}.\nIMPORTANT: Use your tools to retrieve invoice details for this customer ID."
    tools = [get_invoices_by_customer_sorted_by_date, get_invoices_sorted_by_unit_price, get_employee_by_invoice_and_customer]
    msgs = filter_messages(state["messages"])
    try:
        agent = create_react_agent(get_llm(), tools, prompt=sys_prompt)
        res = agent.invoke({"messages": msgs})
        new_msgs = res["messages"][len(msgs):] 
        text_contents = [m.content for m in new_msgs if isinstance(m, AIMessage) and m.content]
        if text_contents:
            combined = "\n\n".join(text_contents)
            combined = re.sub(r'</?function[^>]*>', '', combined)
            combined = '\n'.join([line for line in combined.split('\n') if not line.strip().startswith('=function>') and not line.strip().startswith('</function>')])
            return {"messages": [AIMessage(content=combined.strip(), name="invoice_info")], "visited": ["invoice_info"]}
    except Exception as e:
        print(f"Invoice node tool handling note: {e}")
        db_res = get_invoices_by_customer_sorted_by_date.invoke({"customer_id": str(cid)})
        prompt_fallback = SystemMessage(content=f"You are the Invoice Sub-Agent. Database invoice results for customer {cid}: {db_res}.\nSummarize the invoice information for the user.")
        fallback_ans = get_llm().invoke([prompt_fallback] + msgs[-3:])
        return {"messages": [AIMessage(content=fallback_ans.content.strip(), name="invoice_info")], "visited": ["invoice_info"]}
    return {"messages": [], "visited": ["invoice_info"]}

def create_memory(state: State):
    cid = state.get("customer_id")
    messages = state["messages"]
    new_messages = []
    if messages and isinstance(messages[-1], HumanMessage):
        try:
            sys_msg = SystemMessage(content="You are a polite customer support assistant for Edges Music Store. Respond friendly and concisely.")
            llm = get_fast_llm()
            res = llm.invoke([sys_msg] + messages[-5:])
            new_messages.append(res)
        except Exception as e:
            new_messages.append(AIMessage(content="Hello! How can I help you today?"))
    if not cid:
        return {"messages": new_messages} if new_messages else state
    messages_text = "\n".join([f"{type(m).__name__}: {m.content}" for m in messages[-5:] if isinstance(m, (HumanMessage, AIMessage))])
    try:
        llm = get_fast_llm()
        prompt = f"Extract any new music preferences (favorite artists, genres, etc.) from the conversation. Keep it concise. If none, reply 'None'.\n\n{messages_text}"
        pref = llm.invoke(prompt).content
        if "none" not in pref.lower() and len(pref) > 3:
            existing = LONG_TERM_MEMORY.get(cid, "")
            LONG_TERM_MEMORY[cid] = f"{existing}\n{pref}".strip()
    except Exception as e:
        print(f"Error in memory extraction: {e}")
    return {"messages": new_messages} if new_messages else state

## 6. Build and Compile LangGraph Workflow

In [ ]:
def build_graph():
    workflow = StateGraph(State)
    workflow.add_node("verify_info", verify_info)
    workflow.add_node("ask_human", ask_human)
    workflow.add_node("load_memory", load_memory)
    workflow.add_node("supervisor", supervisor)
    workflow.add_node("music_catalog", music_node)
    workflow.add_node("invoice_info", invoice_node)
    workflow.add_node("create_memory", create_memory)

    workflow.add_edge(START, "verify_info")
    workflow.add_conditional_edges("verify_info", route_after_verification, {"load_memory": "load_memory", "ask_human": "ask_human"})
    workflow.add_edge("ask_human", END)
    workflow.add_edge("load_memory", "supervisor")
    workflow.add_conditional_edges("supervisor", route_supervisor, {"music_catalog": "music_catalog", "invoice_info": "invoice_info", "create_memory": "create_memory"})
    workflow.add_edge("music_catalog", "supervisor")
    workflow.add_edge("invoice_info", "supervisor")
    workflow.add_edge("create_memory", END)

    return workflow

checkpointer = MemorySaver()
app = build_graph().compile(checkpointer=checkpointer)
print("✅ Multi-Agent Graph built and compiled successfully.")

## 7. Project Test Scenarios

--- 
### Testcase 1: "Providing customer information"

In [ ]:
# Testcase 1 - Turn 1
config1 = {"configurable": {"thread_id": "session-test-1"}}
q1 = "My phone number is +55 (12) 3923-5555. How much was my most recent purchase? What albums do you have by the Rolling Stones?"

print(f"👤 USER: {q1}\n")
inputs = {"messages": [HumanMessage(content=q1)]}
for event in app.stream(inputs, config1, stream_mode="values"):
    pass

final_state = app.get_state(config1)
for msg in final_state.values["messages"]:
    if isinstance(msg, AIMessage) and msg.content:
        print(f"🤖 ASSISTANT ({getattr(msg, 'name', 'Assistant')}):\n{msg.content}\n")

In [ ]:
# Testcase 1 - Turn 2 (Preference Recommendation Follow-up)
q2 = "List some songs that match my preferences?"
print(f"👤 USER: {q2}\n")
inputs2 = {"messages": [HumanMessage(content=q2)]}
for event in app.stream(inputs2, config1, stream_mode="values"):
    pass

final_state2 = app.get_state(config1)
new_messages = final_state2.values["messages"][len(final_state.values["messages"]):]
for msg in new_messages:
    if isinstance(msg, AIMessage) and msg.content:
        print(f"🤖 ASSISTANT ({getattr(msg, 'name', 'Assistant')}):\n{msg.content}\n")

--- 
### Testcase 2: "Missing Customer information in new session"

In [ ]:
config2 = {"configurable": {"thread_id": "session-test-2"}}
q3 = "How much was my most recent purchase? What albums do you have by the Rolling Stones?"

print(f"👤 USER: {q3}\n")
inputs3 = {"messages": [HumanMessage(content=q3)]}
for event in app.stream(inputs3, config2, stream_mode="values"):
    pass

state_test2 = app.get_state(config2)
for msg in state_test2.values["messages"]:
    if isinstance(msg, AIMessage) and msg.content:
        print(f"🤖 ASSISTANT:\n{msg.content}\n")